In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [2]:
dir = '/athena/cayuga_0014/scratch/kch4018/abm-files/TME_Simulation/ABM_Jul25/sensitivity_results/parameter_summary.csv'
df = pd.read_csv(dir)

In [6]:
top_10 = df.head(10)
top_10['parameter'].tolist()

['k_TCD8_rec',
 'CancerCell_stemGrowthRate',
 'CancerCell_progGrowthRate',
 'how_exhausted',
 'M1_M2_rec_ratio',
 'TCD8_moveProb',
 'cd8_commitment_delay',
 'k_TCD4_rec',
 'k_Mac_rec',
 'k_PD1_upregulation']

In [1]:
def export_parameter_summary(all_indices,
                             output_dir='sensitivity_results',
                             ST_threshold=0.05):

    rows = []

    for i, param in enumerate(problem['names']):

        mean_st_values = []
        influential_mean_outputs = []
        influential_sd_outputs = []

        for stat, Si in all_indices.items():

            st = Si['ST'][i]

            if np.isnan(st):
                continue

            if stat.endswith('_mean'):

                mean_st_values.append(st)

                if st > ST_threshold:
                    influential_mean_outputs.append(stat)

            elif stat.endswith('_sd'):

                if st > ST_threshold:
                    influential_sd_outputs.append(stat)

        if len(mean_st_values) == 0:
            continue

        rows.append({
            'parameter': param,

            'max_ST': np.max(mean_st_values),
            'mean_ST': np.mean(mean_st_values),
            'median_ST': np.median(mean_st_values),
            'min_ST': np.min(mean_st_values),

            'n_mean_outputs_above_threshold':
                len(influential_mean_outputs),
            'mean_outputs':
                '; '.join(influential_mean_outputs),

            'n_sd_outputs_above_threshold':
                len(influential_sd_outputs),
            'sd_outputs':
                '; '.join(influential_sd_outputs)
        })

    df = pd.DataFrame(rows)

    df = df.sort_values(
        ['n_mean_outputs_above_threshold', 'max_ST'],
        ascending=[False, False]
    )

    outfile = f"{output_dir}/parameter_summary.csv"
    df.to_csv(outfile, index=False)

    print(f"Saved parameter summary to {outfile}")

    return df

In [3]:
from pathlib import Path
import pandas as pd

results_dir = Path("/athena/cayuga_0014/scratch/kch4018/abm-files/TME_Simulation/ABM_Jul25/sensitivity_results")

mean_files = list(results_dir.glob("*_mean_sobol.csv"))

all_st = []

for f in mean_files:

    metric = f.stem.replace("_sobol", "")

    df = pd.read_csv(f)

    tmp = df[["parameter", "ST"]].copy()
    tmp["metric"] = metric

    all_st.append(tmp)

all_st = pd.concat(all_st, ignore_index=True)

all_st.head()

,parameter,ST,metric
0,CancerCell_stemGrowthRate,0.078420,cd8_exhausted_fraction_mean
1,CancerCell_asymDivProb,0.038300,cd8_exhausted_fraction_mean
2,CancerCell_progGrowthRate,0.144660,cd8_exhausted_fraction_mean
3,CancerCell_progDivMax,0.040102,cd8_exhausted_fraction_mean
4,CancerCell_senescentDeathRate,0.042719,cd8_exhausted_fraction_mean


In [4]:
summary = (
    all_st
    .groupby("parameter")
    .agg(
        max_ST=("ST", "max"),
        mean_ST=("ST", "mean"),
        median_ST=("ST", "median"),
        min_ST=("ST", "min")
    )
)

In [5]:
threshold = 0.05

influential = all_st[all_st["ST"] > threshold]

counts = (
    influential
    .groupby("parameter")
    .agg(
        n_mean_outputs_above_threshold=("metric", "count"),
        mean_outputs=("metric",
                      lambda x: "; ".join(x))
    )
)

In [6]:
parameter_summary = (
    summary
    .merge(counts,
           left_index=True,
           right_index=True,
           how="left")
    .fillna({
        "n_mean_outputs_above_threshold": 0,
        "mean_outputs": ""
    })
)

parameter_summary = parameter_summary.sort_values(
    ["n_mean_outputs_above_threshold", "max_ST"],
    ascending=[False, False]
)

In [28]:
pattern = "cd8_exhausted_fraction_mean"
filtered_df = parameter_summary[parameter_summary['mean_outputs'].str.contains(pattern, case=False, na=False)]
filtered_df

,max_ST,mean_ST,median_ST,min_ST,n_mean_outputs_above_threshold,mean_outputs
parameter,,,,,,
CancerCell_progGrowthRate,0.678703,0.374058,0.338302,0.144660,9.0,cd8_exhausted_fraction_mean; cancer_fraction_m...
k_TCD8_rec,0.483348,0.236839,0.154817,0.075216,9.0,cd8_exhausted_fraction_mean; cancer_fraction_m...
CancerCell_stemGrowthRate,0.395192,0.197653,0.149021,0.078420,9.0,cd8_exhausted_fraction_mean; cancer_fraction_m...
how_exhausted,0.534140,0.159122,0.070846,0.034188,7.0,cd8_exhausted_fraction_mean; cancer_fraction_m...
cd8_commitment_delay,0.312906,0.131887,0.058689,0.039918,7.0,cd8_exhausted_fraction_mean; cancer_fraction_m...
M1_M2_rec_ratio,0.223010,0.098450,0.066314,0.035394,7.0,cd8_exhausted_fraction_mean; cd8_naive_fractio...
k_TCD4_rec,0.396420,0.112146,0.053302,0.038684,6.0,cd8_exhausted_fraction_mean; cd8_naive_fractio...
k_Mac_rec,0.328033,0.119719,0.064567,0.028469,6.0,cd8_exhausted_fraction_mean; cd8_naive_fractio...
PDL1_baseline_mean,0.139352,0.071557,0.056399,0.041319,6.0,cd8_exhausted_fraction_mean; cd8_naive_fractio...


In [22]:
parameter_summary.head(15).index.tolist()

['CancerCell_progGrowthRate',
 'k_TCD8_rec',
 'CancerCell_stemGrowthRate',
 'how_exhausted',
 'cd8_commitment_delay',
 'TCD8_moveProb',
 'M1_M2_rec_ratio',
 'k_TCD4_rec',
 'k_Mac_rec',
 'PDL1_baseline_mean',
 'TCD4_death_rate',
 'k_PD1_upregulation',
 'k_TCR_activation',
 'k_PD1_signaling',
 'k_TCD4_div']

In [21]:
top_10 = df.head(15)
top_10['parameter'].tolist()

['CancerCell_stemGrowthRate',
 'CancerCell_asymDivProb',
 'CancerCell_progGrowthRate',
 'CancerCell_progDivMax',
 'CancerCell_senescentDeathRate',
 'CancerCell_stemMoveProb',
 'CancerCell_MoveProb',
 'cd8_commitment_delay',
 'p_memory_diff',
 'TCD8_div_Interval',
 'TCD8_div_limit_effector',
 'TCD8_div_limit_memory',
 'k_TCR_activation',
 'k_cytokine_CD8',
 'k_hill_MDSC_TCD8']